In [31]:
import pandas as pd
import numpy as np
import faiss
import pickle
import joblib
import anthropic
from sentence_transformers import SentenceTransformer

# Load model and scaler
ridge = joblib.load('../models/ridge_final.pkl')
scaler = joblib.load('../models/scaler.pkl')

# Load vector store
index = faiss.read_index('../data/vector_store/houses.index')
with open('../data/vector_store/documents.pkl', 'rb') as f:
    documents = pickle.load(f)

# Load embedder
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Load feature columns so we know what the model expects
X_train = pd.read_csv('../data/processed/X_train.csv')
feature_cols = X_train.columns.tolist()

print("Everything loaded")
print(f"Model features: {len(feature_cols)}")
print(f"Vector store size: {index.ntotal}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Everything loaded
Model features: 211
Vector store size: 2927


In [32]:
from google import genai

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

In [33]:
def predict_price(input_features: dict) -> float:
    """
    Takes a dict of raw house features and returns predicted price in dollars.
    input_features should match the cleaned dataset columns before encoding.
    """
    # Create a template row from training data means
    template = pd.read_csv('../data/processed/cleaned.csv')
    row = template.mean(numeric_only=True).to_dict()
    
    # Override with user inputs
    for key, val in input_features.items():
        row[key] = val
    
    # Build full feature vector matching training columns
    row_df = pd.DataFrame([row])
    
    # One-hot encode to match training shape
    cat_cols = template.select_dtypes(include='str').columns.tolist()
    row_df = pd.get_dummies(row_df)
    
    # Align columns with training data
    row_df = row_df.reindex(columns=feature_cols, fill_value=0)
    
    # Scale
    numeric_cols = X_train.select_dtypes(include='number').columns.tolist()
    row_df[numeric_cols] = scaler.transform(row_df[numeric_cols])
    
    # Predict and convert from log scale
    log_price = ridge.predict(row_df)[0]
    return float(np.expm1(log_price))

In [34]:
def retrieve_comps(query: str, k: int = 5) -> list:
    """
    Takes a natural language query and returns k most similar house sales.
    """
    query_embedding = embedder.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, k=k)
    
    comps = []
    for i, idx in enumerate(indices[0]):
        comps.append({
            'rank': i + 1,
            'distance': float(distances[0][i]),
            'description': documents[idx]
        })
    
    return comps

In [ ]:
def generate_negotiation_strategy(
    property_description: str,
    predicted_price: float,
    asking_price: float,
    comps: list
) -> str:

    comps_text = "\n\n".join([
        f"Comp {c['rank']}: {c['description']}" 
        for c in comps
    ])
    
    prompt = f"""You are an expert real estate negotiation advisor.

A buyer is considering purchasing the following property:
{property_description}

Asking price: ${asking_price:,.0f}
ML model predicted fair value: ${predicted_price:,.0f}
Price difference: ${asking_price - predicted_price:,.0f} ({'above' if asking_price > predicted_price else 'below'} predicted value)

Here are the 5 most comparable recent sales in the area:
{comps_text}

Based on the predicted value and these comparable sales, provide:
1. A recommended offer price with justification
2. Key negotiation leverage points specific to these comps
3. A suggested negotiation strategy (initial offer, concessions to request, walk-away price)
4. Any red flags or strengths in this property's pricing

Be specific and ground every recommendation in the comparable sales data provided.
Do not give generic advice - every point must reference the actual numbers."""

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    
    return response.text

In [36]:
# Define a test property
test_property = {
    'Gr Liv Area': 1800,
    'Overall Qual': 7,
    'Year Built': 1995,
    'Full Bath': 2,
    'Bedroom AbvGr': 3,
    'Garage Cars': 2,
    'Total Bsmt SF': 900,
}

asking_price = 220000

# Step 1 - predict price
predicted_price = predict_price(test_property)
print(f"Predicted fair value: ${predicted_price:,.0f}")
print(f"Asking price: ${asking_price:,.0f}")
print(f"Difference: ${asking_price - predicted_price:,.0f}")

# Step 2 - retrieve comps
query = f"3 bedroom house 1800 sq ft Overall Quality 7 built 1995 2 car garage"
comps = retrieve_comps(query, k=5)

print(f"\nRetrieved {len(comps)} comps")
for c in comps:
    print(f"\nComp {c['rank']}:")
    print(c['description'])

Predicted fair value: $83,742
Asking price: $220,000
Difference: $136,258

Retrieved 5 comps

Comp 1:
A 1053 sq ft home in the NAmes 
    neighborhood sold for $142,100. The property has an Overall Quality 
    rating of 5 out of 10, built in 1963. 
    It features 1 full bathrooms, 3 
    bedrooms, and a 2-car garage. 
    The kitchen quality is Good and the 
    basement size is 1053 sq ft.

Comp 2:
A 1053 sq ft home in the NAmes 
    neighborhood sold for $145,500. The property has an Overall Quality 
    rating of 5 out of 10, built in 1959. 
    It features 1 full bathrooms, 3 
    bedrooms, and a 1-car garage. 
    The kitchen quality is Good and the 
    basement size is 1053 sq ft.

Comp 3:
A 1337 sq ft home in the NAmes 
    neighborhood sold for $177,000. The property has an Overall Quality 
    rating of 6 out of 10, built in 1997. 
    It features 2 full bathrooms, 2 
    bedrooms, and a 2-car garage. 
    The kitchen quality is Good and the 
    basement size is 1405 sq ft

In [37]:
property_description = f"""
3 bedroom, 2 bathroom home
1,800 sq ft above grade living area
Built in 1995, Overall Quality 7/10
2-car garage, 900 sq ft basement
"""

strategy = generate_negotiation_strategy(
    property_description=property_description,
    predicted_price=predicted_price,
    asking_price=asking_price,
    comps=comps
)

print("NEGOTIATION STRATEGY")
print("=" * 50)
print(strategy)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 4.778145898s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.goo